# mini-qforest — 03. Hybrid quantum classifiers

**Question:** can a small quantum circuit, simulated on a classical computer, tell forest from non-forest as well as a classical model of the same size?

**Method (hybrid = classical pre-processing + quantum model + classical optimisation):**
1. a balanced subsample: 200 training and 200 test images (simulating circuits is slow);
2. PCA reduces the 35 features to **4 dimensions = 4 qubits**;
3. two quantum models:
   - a **variational quantum classifier (VQC)**: angle encoding + trainable entangling layers, 37 parameters;
   - a **quantum kernel SVM**: an IQP feature map defines the similarity between two images;
4. classical baselines on the *same 4 dimensions*: SVM (RBF), logistic regression, an MLP with 37 parameters, and a random forest on all 35 features as a reference.

> Run notebook 01 first. No GPU needed; the quantum kernel cell takes a few minutes.

In [ ]:
!pip install -q pennylane

In [ ]:
# --- Setup: works in Google Colab (results saved to your Google Drive) or locally ---
import os, sys, time
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/mini-qforest"
else:
    WORKDIR = os.path.abspath("..")
DATA_DIR = os.path.join(WORKDIR, "data")
RES_DIR = os.path.join(WORKDIR, "results")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)
SEED = 42
print("Working directory:", WORKDIR)

In [ ]:
# --- Helper: every model writes its score to results/results.csv (shared by the 3 notebooks) ---
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

RESULTS_CSV = os.path.join(RES_DIR, "results.csv")

def log_result(model, task, y_true, y_pred, n_params=None, train_time_s=None):
    binary = len(np.unique(y_true)) == 2
    row = {
        "model": model,
        "task": task,
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "f1": round(f1_score(y_true, y_pred, average="binary" if binary else "macro"), 4),
        "n_params": n_params,
        "train_time_s": None if train_time_s is None else round(train_time_s, 1),
    }
    df = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame(columns=list(row))
    df = df[~((df["model"] == model) & (df["task"] == task))]
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(RESULTS_CSV, index=False)
    print(row)
    return row

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler

d = np.load(os.path.join(RES_DIR, "features.npz"), allow_pickle=True)
X_all, ybin = d["X"], d["ybin"]
s = np.load(os.path.join(RES_DIR, "split.npz"))

rng = np.random.default_rng(SEED)
def balanced(ids, n_per_class):
    return np.concatenate([rng.choice(ids[ybin[ids] == c], n_per_class, replace=False) for c in (0, 1)])

tr = balanced(s["train"], 100)   # 200 training images, 50 % forest
te = balanced(s["test"], 100)    # 200 test images, taken from the common test set
y_tr, y_te = ybin[tr], ybin[te]

N_QUBITS = 4
std = StandardScaler().fit(X_all[tr])
pca = PCA(n_components=N_QUBITS, random_state=SEED).fit(std.transform(X_all[tr]))
scale = MinMaxScaler(feature_range=(0, np.pi)).fit(pca.transform(std.transform(X_all[tr])))
prep = lambda ids: np.clip(scale.transform(pca.transform(std.transform(X_all[ids]))), 0, np.pi)
X_tr, X_te = prep(tr), prep(te)
print("explained variance of the 4 components:", pca.explained_variance_ratio_.round(3),
      "total", pca.explained_variance_ratio_.sum().round(3))

TASK = "forest_vs_rest (balanced subsample, 4 PCA dims)"

In [ ]:
plt.figure(figsize=(5, 4))
plt.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap="Greens", edgecolor="k", s=25)
plt.xlabel("PC1 (scaled to [0, π])"); plt.ylabel("PC2"); plt.title("Training images: forest in dark green")
plt.tight_layout(); plt.show()

## 1. Variational quantum classifier (VQC)

- **Encoding:** each of the 4 features becomes the rotation angle of one qubit (`AngleEmbedding`, RY rotations).
- **Trainable part:** 3 `StronglyEntanglingLayers` = 3 × 4 × 3 = 36 rotation angles, plus CNOT gates that entangle the qubits.
- **Output:** expectation value of Pauli-Z on qubit 0, in [−1, 1], plus a trainable bias. Prediction = its sign.
- **Training:** square loss, Adam optimiser, gradients computed exactly by the simulator.

In [ ]:
import pennylane as qml
from pennylane import numpy as pnp

N_LAYERS = 3
dev = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev)
def circuit(weights, x):
    qml.AngleEmbedding(x, wires=range(N_QUBITS), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return qml.expval(qml.PauliZ(0))

def vqc(weights, bias, X):
    return circuit(weights, X) + bias          # X can be a whole batch (parameter broadcasting)

shape = qml.StronglyEntanglingLayers.shape(n_layers=N_LAYERS, n_wires=N_QUBITS)
print(qml.draw(circuit)(np.zeros(shape), X_tr[0]))

In [ ]:
np.random.seed(SEED)
weights = pnp.array(0.1 * np.random.randn(*shape), requires_grad=True)
bias = pnp.array(0.0, requires_grad=True)
n_params_vqc = weights.size + 1
Y_tr = 2 * y_tr - 1                              # labels in {-1, +1}

def cost(w, b, X, Y):
    return pnp.mean((Y - vqc(w, b, X)) ** 2)

opt = qml.AdamOptimizer(stepsize=0.05)
history = []
t0 = time.time()
for epoch in range(30):
    perm = rng.permutation(len(X_tr))
    for start in range(0, len(X_tr), 20):
        bi = perm[start:start + 20]
        (weights, bias), _ = opt.step_and_cost(lambda w, b: cost(w, b, X_tr[bi], Y_tr[bi]), weights, bias)
    acc = (np.sign(vqc(weights, bias, X_tr)) == Y_tr).mean()
    history.append((float(cost(weights, bias, X_tr, Y_tr)), acc))
    if (epoch + 1) % 5 == 0:
        print(f"epoch {epoch + 1:2d}  cost {history[-1][0]:.3f}  train acc {acc:.3f}")
dt_vqc = time.time() - t0

pred_vqc = (np.sign(vqc(weights, bias, X_te)) > 0).astype(int)
log_result(f"VQC ({N_QUBITS} qubits, {N_LAYERS} layers)", TASK, y_te, pred_vqc, n_params_vqc, dt_vqc)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot([h[0] for h in history], label="cost"); ax.plot([h[1] for h in history], label="train accuracy")
ax.set_xlabel("epoch"); ax.legend(); ax.grid(alpha=0.3); ax.set_title("VQC training")
plt.tight_layout(); plt.savefig(os.path.join(RES_DIR, "vqc_training.png"), dpi=150); plt.show()

## 2. Quantum kernel SVM

The kernel between two images is the overlap of their quantum states: k(x, x′) = |⟨φ(x′)|φ(x)⟩|².
It is measured as the probability of returning to |0000⟩ after encoding *x* then *un-encoding* *x′*.
The **IQP** feature map entangles the qubits, so the kernel is harder to reproduce classically than a simple angle encoding.

In [ ]:
dev_k = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev_k)
def overlap(x1, x2):
    qml.IQPEmbedding(x1, wires=range(N_QUBITS), n_repeats=2)
    qml.adjoint(qml.IQPEmbedding)(x2, wires=range(N_QUBITS), n_repeats=2)
    return qml.probs(wires=range(N_QUBITS))

def qkernel(x1, x2):
    return overlap(x1, x2)[0]

t0 = time.time()
K_tr = qml.kernels.square_kernel_matrix(X_tr, qkernel, assume_normalized_kernel=True)
K_te = qml.kernels.kernel_matrix(X_te, X_tr, qkernel)
dt_k = time.time() - t0
print(f"kernel matrices computed in {dt_k:.0f} s")

from sklearn.svm import SVC
qsvm = SVC(kernel="precomputed", C=1.0).fit(K_tr, y_tr)
log_result("Quantum kernel SVM (IQP, 4 qubits)", TASK, y_te, qsvm.predict(K_te), None, dt_k)

In [ ]:
plt.figure(figsize=(4.5, 4))
order = np.argsort(y_tr)
plt.imshow(np.asarray(K_tr)[np.ix_(order, order)], cmap="viridis"); plt.colorbar(fraction=0.046)
plt.title("Quantum kernel (sorted: non-forest, then forest)"); plt.tight_layout()
plt.savefig(os.path.join(RES_DIR, "quantum_kernel.png"), dpi=150); plt.show()

## 3. Classical baselines on the same data

In [ ]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier

baselines = [
    ("SVM RBF (same 4 dims)", SVC(kernel="rbf", C=1.0), X_tr, X_te),
    ("Logistic regression (same 4 dims)", LogisticRegression(max_iter=1000), X_tr, X_te),
    ("MLP 4-6-1, 37 params (same 4 dims)", MLPClassifier(hidden_layer_sizes=(6,), max_iter=5000, random_state=SEED), X_tr, X_te),
    ("Random forest (all 35 features, reference)", RandomForestClassifier(300, random_state=SEED), X_all[tr], X_all[te]),
]
for name, clf, A, B in baselines:
    t0 = time.time(); clf.fit(A, y_tr); dt = time.time() - t0
    n_p = sum(c.size for c in clf.coefs_) + sum(c.size for c in clf.intercepts_) if hasattr(clf, "coefs_") else None
    log_result(name, TASK, y_te, clf.predict(B), n_p, dt)

## 4. All results from the three notebooks

In [ ]:
res = pd.read_csv(RESULTS_CSV)
display(res.sort_values(["task", "accuracy"], ascending=[True, False]))

sub = res[res["task"] == TASK].sort_values("accuracy")
colors = ["#6a1b9a" if ("VQC" in m or "Quantum" in m) else "#9e9e9e" for m in sub["model"]]
plt.figure(figsize=(8, 3.8))
plt.barh(sub["model"], sub["accuracy"], color=colors)
plt.xlim(0.5, 1.0); plt.xlabel("test accuracy (200 balanced images)")
plt.title("Forest vs rest — quantum models in purple")
plt.tight_layout(); plt.savefig(os.path.join(RES_DIR, "comparison_quantum.png"), dpi=150); plt.show()

## Discussion (to complete with your results)

- Do the quantum models reach the level of the classical models **with the same 4 inputs and a similar number of parameters**? A tie or a loss is a valid result: what matters is a fair comparison.
- What does the reduction to 4 dimensions cost? Compare with the random forest on all 35 features.
- Limits: noiseless simulation, small sample, only 4 qubits. On real hardware (e.g. IBM Quantum), noise would lower accuracy.

**Link with the PhD (Q-FOREST):** the thesis goes further by combining both ideas — **graphs and quantum circuits**.
Possible directions: encoding the graph structure in the circuit (layers inspired by QAOA, whose entangling gates follow the edges of the graph), quantum kernels between graphs, or a GNN whose aggregation step is a parameterised circuit. Key question: on which part of the pipeline can a quantum model bring a measurable benefit, at what cost?